# Week 8 Exercise — Building Agents with Open Source Models

Using:
- **OpenRouter** for 3 open-source models: **OpenAI gpt-oss-120b**, **DeepSeek**, **Llama 3.1 70B** (pricing + scanning).
- **RAG** with Chroma + SentenceTransformer.
- **Scanner**: RSS deals → OpenRouter selects 5 deals with clear prices.
- **Pricer**: RAG + 3 OpenRouter models → consensus (average).
- **Messaging**: **Pushover** push notifications.
- **Gradio** UI: Run Scan, table of opportunities, row click to notify.

Set `OPENROUTER_API_KEY`, `HF_TOKEN`, and optionally `PUSHOVER_USER` / `PUSHOVER_TOKEN` in `.env`.

## 1. Imports and environment

In [18]:
import os
import re
import json
import logging
import time
from pathlib import Path
from typing import List, Dict, Optional

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
import chromadb
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import feedparser
from bs4 import BeautifulSoup
import requests
from tqdm.notebook import tqdm
import gradio as gr


In [19]:
load_dotenv(override=True)

OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"

assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY in .env"
print("OPENROUTER_API_KEY: set")
print("PUSHOVER_USER:", "set" if os.getenv("PUSHOVER_USER") else "not set")
print("PUSHOVER_TOKEN:", "set" if os.getenv("PUSHOVER_TOKEN") else "not set")

OPENROUTER_API_KEY: set
PUSHOVER_USER: set
PUSHOVER_TOKEN: set


## 2. OpenRouter model IDs (3 open-source models)

In [20]:
MODEL_SCANNER = "deepseek/deepseek-chat"
MODEL_PRICER_1 = "openai/gpt-oss-120b"
MODEL_PRICER_2 = "deepseek/deepseek-chat"
MODEL_PRICER_3 = "meta-llama/llama-3.1-70b-instruct"
PRICER_MODELS = [MODEL_PRICER_1, MODEL_PRICER_2, MODEL_PRICER_3]

def openrouter_client():
    return OpenAI(base_url=OPENROUTER_BASE, api_key=OPENROUTER_API_KEY)

client = openrouter_client()
print("Scanner:", MODEL_SCANNER)
print("Pricers:", PRICER_MODELS)

Scanner: deepseek/deepseek-chat
Pricers: ['openai/gpt-oss-120b', 'deepseek/deepseek-chat', 'meta-llama/llama-3.1-70b-instruct']


## 3. Inline data models and RSS scraping

In [21]:
class Deal(BaseModel):
    product_description: str = Field(description="Summary of the product")
    price: float = Field(description="Actual price")
    url: str = Field(description="Deal URL")

class DealSelection(BaseModel):
    deals: List[Deal] = Field(description="List of 5 deals")

class Opportunity(BaseModel):
    deal: Deal
    estimate: float
    discount: float

RSS_FEEDS = [
    "https://www.dealnews.com/c142/Electronics/?rss=1",
    "https://www.dealnews.com/c39/Computers/?rss=1",
    "https://www.dealnews.com/f1912/Smart-Home/?rss=1",
]

def extract_html(html_snippet: str) -> str:
    soup = BeautifulSoup(html_snippet, "html.parser")
    div = soup.find("div", class_="snippet summary")
    if div:
        text = div.get_text(strip=True)
        text = re.sub(r"<[^<]+?>", "", text)
        return text.strip().replace("\n", " ")[:500]
    return html_snippet[:500].replace("\n", " ")

class ScrapedDeal:
    def __init__(self, entry: Dict):
        self.title = entry.get("title", "")[:100]
        self.summary = extract_html(entry.get("summary", ""))
        self.url = entry.get("links", [{}])[0].get("href", "")
        try:
            r = requests.get(self.url, timeout=10)
            soup = BeautifulSoup(r.content, "html.parser")
            content_div = soup.find("div", class_="content-section")
            content = content_div.get_text() if content_div else ""
            content = content.replace("\nmore", "").replace("\n", " ")[:500]
        except Exception:
            content = ""
        self.details = content

    def describe(self) -> str:
        return f"Title: {self.title}\nDetails: {self.details}\nURL: {self.url}"

def fetch_scraped_deals(show_progress: bool = True) -> List[ScrapedDeal]:
    out = []
    it = tqdm(RSS_FEEDS) if show_progress else RSS_FEEDS
    for feed_url in it:
        feed = feedparser.parse(feed_url)
        for entry in feed.entries[:10]:
            try:
                out.append(ScrapedDeal(entry))
            except Exception:
                pass
            time.sleep(0.05)
    return out

## 4. Load catalog from HuggingFace Hub

In [22]:
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN", ""), add_to_git_credential=False)

DATA_USER = "ed-donner"
LITE = True
dataset_name = f"{DATA_USER}/items_lite" if LITE else f"{DATA_USER}/items_full"
ds = load_dataset(dataset_name)
train_rows = list(ds["train"])

def summary(row):
    return (row.get("summary") or row.get("title") or "")[:500]

print("Train rows:", len(train_rows))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Train rows: 20000


## 5. Build Chroma vector store (RAG)

In [23]:
NOTEBOOK_DIR = Path.cwd()
DB_PATH = NOTEBOOK_DIR / "spotlight_vectorstore"
DB_PATH.mkdir(exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(DB_PATH))
coll_name = "products"
if coll_name in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(coll_name)
collection = chroma_client.create_collection(coll_name)

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
CAP = 20_000
batch_size = 1000

for i in tqdm(range(0, min(CAP, len(train_rows)), batch_size)):
    batch = train_rows[i : i + batch_size]
    docs = [summary(r) for r in batch]
    vecs = encoder.encode(docs).astype(float).tolist()
    metas = [{"category": r.get("category", ""), "price": float(r.get("price", 0))} for r in batch]
    ids = [f"doc_{j}" for j in range(i, i + len(batch))]
    collection.add(ids=ids, documents=docs, embeddings=vecs, metadatas=metas)

print("Chroma count:", collection.count())

  0%|          | 0/20 [00:00<?, ?it/s]

Chroma count: 20000


## 6. RAG + OpenRouter pricer (3 models, consensus = average)

In [24]:
def find_similars(description: str, n: int = 5):
    vec = encoder.encode([description])
    res = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=n)
    docs = res["documents"][0]
    prices = [m["price"] for m in res["metadatas"][0]]
    return docs, prices

def make_rag_context(similars: List[str], prices: List[float]) -> str:
    ctx = "Similar products and prices:\n"
    for s, p in zip(similars, prices):
        ctx += f"- {s[:200]}... Price ${p:.2f}\n"
    return ctx

In [25]:
def parse_price_from_reply(text: str) -> float:
    text = str(text).replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0

def price_with_one_model(description: str, context: str, model: str) -> float:
    prompt = f"Estimate the price of this product in USD. Reply with one number only.\n\nProduct:\n{description}\n\n{context}"
    try:
        r = client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
        return parse_price_from_reply(r.choices[0].message.content or "0")
    except Exception:
        return 0.0

In [26]:
def consensus_price(description: str) -> float:
    docs, prices = find_similars(description)
    context = make_rag_context(docs, prices)
    estimates = [price_with_one_model(description, context, m) for m in PRICER_MODELS]
    estimates = [e for e in estimates if e > 0]
    return sum(estimates) / len(estimates) if estimates else 0.0

## 7. Scanner: fetch deals + OpenRouter to select 5 deals (JSON)

In [27]:
SCANNER_SYSTEM = """You select the 5 best deals: most detailed product description and clearest price.
Respond with valid JSON only, no markdown: {"deals": [{"product_description": "...", "price": 99.99, "url": "https://..."}, ...]}
Price must be the actual product price (number). Include exactly 5 deals. Skip deals where price is unclear."""

In [28]:
def scan_deals(seen_urls: List[str]) -> Optional[DealSelection]:
    scraped = fetch_scraped_deals(show_progress=False)
    scraped = [s for s in scraped if s.url not in seen_urls]
    if not scraped:
        return None
    user_content = "Deals:\n" + "\n\n".join([s.describe() for s in scraped[:30]])
    user_content += "\n\nReturn JSON with key 'deals' (array of 5 items with product_description, price, url)."
    try:
        r = client.chat.completions.create(
            model=MODEL_SCANNER,
            messages=[{"role": "system", "content": SCANNER_SYSTEM}, {"role": "user", "content": user_content}],
        )
        raw = (r.choices[0].message.content or "{}").strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```\w*\n?", "", raw).rstrip("`")
        data = json.loads(raw)
        deals = [Deal(**d) for d in data.get("deals", []) if d.get("price", 0) > 0][:5]
        return DealSelection(deals=deals) if deals else None
    except Exception as e:
        logging.warning("Scanner parse error: %s", e)
        return None

## 8. Pushover messaging (inline)

In [30]:
def pushover_push(message: str) -> bool:
    user = os.getenv("PUSHOVER_USER")
    token = os.getenv("PUSHOVER_TOKEN")
    if not user or not token:
        return False
    try:
        requests.post(PUSHOVER_URL, data={"user": user, "token": token, "message": message[:1024], "sound": "cashregister"}, timeout=5)
        return True
    except Exception:
        return False

def alert_opportunity(opp: Opportunity) -> bool:
    msg = f"Deal! Price=${opp.deal.price:.2f} Estimate=${opp.estimate:.2f} Discount=${opp.discount:.2f} {opp.deal.product_description[:50]}... {opp.deal.url}"
    return pushover_push(msg)

DEAL_THRESHOLD = 50
print("Pushover ready (set PUSHOVER_USER and PUSHOVER_TOKEN to enable)")

Pushover ready (set PUSHOVER_USER and PUSHOVER_TOKEN to enable)


## 9. Planning: scan -> price -> best opportunity -> notify

In [31]:
def run_plan(seen_urls: List[str]) -> tuple[Optional[Opportunity], List[str], List[Opportunity]]:
    selection = scan_deals(seen_urls)
    if not selection or not selection.deals:
        return None, seen_urls, []
    opportunities = []
    for d in selection.deals:
        est = consensus_price(d.product_description)
        discount = est - d.price
        opportunities.append(Opportunity(deal=d, estimate=est, discount=discount))
    opportunities.sort(key=lambda o: o.discount, reverse=True)
    best = opportunities[0] if opportunities else None
    new_memory = list(seen_urls)
    if best:
        new_memory.append(best.deal.url)
        if best.discount > DEAL_THRESHOLD:
            alert_opportunity(best)
    return best, new_memory, opportunities

## 10. Gradio UI (table + Run Scan, row click = push)

In [32]:
def table_from_opportunities(opps: List[Opportunity]) -> List[List[str]]:
    return [[o.deal.product_description[:80] + "...", f"${o.deal.price:.2f}", f"${o.estimate:.2f}", f"${o.discount:.2f}", o.deal.url] for o in opps]

def run_scan_click(memory_list, opps_list):
    seen = memory_list or []
    best, new_memory, all_opps = run_plan(seen)
    table = table_from_opportunities(all_opps)
    new_opps = (opps_list or []) + all_opps
    status = f"Best discount: ${best.discount:.2f}" if best else "No deals this run."
    if best and best.discount > DEAL_THRESHOLD:
        status += " Push sent."
    return table, new_memory, new_opps, all_opps, status

def on_row_select(current_run_opps, evt: gr.SelectData):
    if not current_run_opps or evt.index[0] >= len(current_run_opps):
        return "No row selected."
    alert_opportunity(current_run_opps[evt.index[0]])
    return "Push sent for selected deal."

In [33]:
with gr.Blocks(title="The Price is Right (Open source models + Pushover)", fill_width=True) as ui:
    gr.Markdown("# The Price is Right - Open Source Models (gpt-oss-120b, DeepSeek, Llama) + Pushover")
    gr.Markdown("Scan RSS -> price with 3 models + RAG -> notify via Pushover.")

    memory_state = gr.State([])
    opps_state = gr.State([])
    current_run_opps = gr.State([])

    dataframe = gr.Dataframe(headers=["Description", "Price", "Estimate", "Discount", "URL"], label="Opportunities", wrap=True, row_count=10)
    status_box = gr.Textbox(label="Status", interactive=False)
    scan_btn = gr.Button("Run Scan", variant="primary")
    row_click_status = gr.Textbox(label="Row action", interactive=False)

    scan_btn.click(fn=run_scan_click, inputs=[memory_state, opps_state], outputs=[dataframe, memory_state, opps_state, current_run_opps, status_box])
    dataframe.select(fn=on_row_select, inputs=[current_run_opps], outputs=[row_click_status])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
